# FTI posterior, likelihood trace, corner plot, and maximum-likelihood values

FTI-inclusive, config-consistent version using the lisabeta output naming convention.

This notebook:

1. Reads every FTI JSON file in the selected config directory.
2. Uses the sampled coordinates from each JSON `infer_params`.
3. Forces the active FTI parameter into every full, small, and focused corner plot.
4. Uses the processed posterior only when it contains the active FTI parameter.
5. Falls back to the raw chain when the processed posterior omits the FTI parameter.
6. Prints the exact source and parameter list used for every corner plot.
7. Produces traces, likelihood plots, diagnostics, consistency checks, and maximum-likelihood tables.


In [1]:
import json
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import lisabeta.utils.plotutils as plotutils

try:
    import corner
    HAS_CORNER = True
except ImportError:
    HAS_CORNER = False
    print("corner is not installed. Raw fallback corner plots will not work.")

from IPython.display import display


/pbs/home/a/amahdi/lisabeta/src/lisabeta/waveforms/bbh/lalsim_wrap.py:10: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


## Select the run/config directory

`CASE_ID` is set to `M1e6_SEOBv5_id_3` to match the confirmed example path. The notebook processes every `.json` file inside that directory, so no FTI parameter is hard-coded.


In [2]:
PE_BASE_DIR = Path('/pbs/home/a/amahdi/PE_for_my_work')
CASE_ID = 'M1e6_SEOBv5_id_2'

config_dir = PE_BASE_DIR / 'config_files' / CASE_ID
config_files = sorted(config_dir.glob('*.json'))

print('PE base directory:', PE_BASE_DIR)
print('Case ID:', CASE_ID)
print('Config directory:', config_dir)
print('Number of JSON config files found:', len(config_files))

if not config_files:
    raise FileNotFoundError(
        f'No JSON files were found in {config_dir}. '
        'Check PE_BASE_DIR and CASE_ID.'
    )

for f in config_files:
    print(' ', f.name)

plot_output_dir = config_dir / 'fti_plots_all_parameters_concat_path'
corner_output_dir = plot_output_dir / 'corner'
trace_output_dir = plot_output_dir / 'trace'
likelihood_output_dir = plot_output_dir / 'likelihood'
table_output_dir = plot_output_dir / 'tables'

for d in [plot_output_dir, corner_output_dir, trace_output_dir, likelihood_output_dir, table_output_dir]:
    d.mkdir(parents=True, exist_ok=True)

print('Plot output directory:', plot_output_dir)

BAD_LNL_CUT = -1e12
MAX_RAW_SAMPLES_FOR_CORNER = 20000
RANDOM_SEED = 1234

# Actual sampled intrinsic parameters in the current config creator.
SAMPLED_INTRINSIC_PARAMS = ['Mchirp', 'q', 'chiPN', 'chim']
SAMPLED_EXTENDED_PARAMS = ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'lambda', 'beta']


PE base directory: /pbs/home/a/amahdi/PE_for_my_work
Case ID: M1e6_SEOBv5_id_2
Config directory: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2
Number of JSON config files found: 12
  M1e6_SEOBv5_id_2_fti_dchi0v_IMR.json
  M1e6_SEOBv5_id_2_fti_dchi1v_IMR.json
  M1e6_SEOBv5_id_2_fti_dchi2v_IMR.json
  M1e6_SEOBv5_id_2_fti_dchi3v_IMR.json
  M1e6_SEOBv5_id_2_fti_dchi4v_IMR.json
  M1e6_SEOBv5_id_2_fti_dchi5lv_IMR.json
  M1e6_SEOBv5_id_2_fti_dchi6lv_IMR.json
  M1e6_SEOBv5_id_2_fti_dchi6v_IMR.json
  M1e6_SEOBv5_id_2_fti_dchi7v_IMR.json
  M1e6_SEOBv5_id_2_fti_dchiMinus2v_IMR.json
  M1e6_SEOBv5_id_2_fti_dkappa_S_IMR.json
  M1e6_SEOBv5_id_2_fti_dxi0v_IMR.json
Plot output directory: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path


## Helper functions

The confirmed output is created by direct string concatenation of `out_dir` and `out_name`. Therefore, the notebook searches these paths first:

1. `Path(str(out_dir) + f"{out_name}.h5")`
2. `Path(str(out_dir) + f"{out_name}_raw.h5")`

For the example run, this gives a path of the form:

`run_data/M1e6_SEOBv5_id_3M1e6_SEOBv5_id_3_fti_dchi0v_IMR_raw.h5`

Ordinary joined-directory paths are retained only as compatibility fallbacks.


In [3]:
def infer_pe_base_dir_from_config_location(subfile):
    subfile = Path(subfile)
    config_case_dir = subfile.parent
    config_files_dir = config_case_dir.parent
    return config_files_dir.parent


def infer_run_data_root_from_config_location(subfile):
    return infer_pe_base_dir_from_config_location(subfile) / 'run_data'


def infer_run_dir_from_config_location(subfile):
    subfile = Path(subfile)
    return infer_run_data_root_from_config_location(subfile) / subfile.parent.name


def unique_paths(paths):
    out = []
    seen = set()
    for label, p in paths:
        p = Path(p)
        key = str(p)
        if key not in seen:
            out.append((label, p))
            seen.add(key)
    return out


def candidate_output_paths(subfile, out_dir, out_name, suffix):
    """
    Return output candidates in the order appropriate for these runs.

    The first two candidates reproduce the confirmed lisabeta string-concatenated
    naming convention. The final two are ordinary Path joins retained as fallbacks.
    """
    subfile = Path(subfile)
    out_dir = Path(out_dir)
    inferred_run_data_root = infer_run_data_root_from_config_location(subfile)
    inferred_run_dir = infer_run_dir_from_config_location(subfile)
    case_id = subfile.parent.name

    candidates = [
        # Confirmed pattern from run_params['out_dir'] and run_params['out_name'].
        ('configured_string_concat', Path(str(out_dir) + f'{out_name}{suffix}')),

        # Same confirmed pattern inferred from the config directory location.
        ('inferred_string_concat', inferred_run_data_root / f'{case_id}{out_name}{suffix}'),

        # Compatibility fallbacks for outputs written with normal directory joins.
        ('configured_path_join', out_dir / f'{out_name}{suffix}'),
        ('inferred_path_join', inferred_run_dir / f'{out_name}{suffix}'),
    ]

    return unique_paths(candidates)


def first_existing_labeled_path(candidates):
    for label, p in candidates:
        if p.exists():
            return label, p
    return None, None


def detect_active_fti_param(config):
    infer_params = list(config.get('prior_params', {}).get('infer_params', []))
    fti_params = config.get('waveform_params', {}).get('tgr_params', {}).get('FTI', {}).get('fti_params', [])

    active = [p for p in infer_params if p in fti_params or p.startswith('fti_')]
    if active:
        return active[-1]
    if infer_params:
        return infer_params[-1]
    return None


def get_case_info(subfile):
    subfile = Path(subfile)

    with open(subfile, 'r') as f:
        config = json.load(f)

    prior_params = config.get('prior_params', {})
    run_params = config.get('run_params', {})

    infer_params = list(prior_params.get('infer_params', []))
    list_params = list(prior_params.get('list_params', []))
    fisher_params = list(run_params.get('init_fisher_options', {}).get('fisher_params', []))
    fti_param = detect_active_fti_param(config)

    out_dir = Path(run_params.get('out_dir', ''))
    out_name = run_params.get('out_name', subfile.stem)

    post_candidates = candidate_output_paths(subfile, out_dir, out_name, '.h5')
    raw_candidates = candidate_output_paths(subfile, out_dir, out_name, '_raw.h5')

    post_path_kind, postfile = first_existing_labeled_path(post_candidates)
    raw_path_kind, post_raw_file = first_existing_labeled_path(raw_candidates)

    warnings = []
    if infer_params != list_params:
        warnings.append('infer_params and list_params differ.')
    if fisher_params and fisher_params != infer_params:
        warnings.append('init_fisher_options.fisher_params and infer_params differ.')
    if fti_param is None:
        warnings.append('Could not detect active FTI parameter.')
    elif fti_param not in infer_params:
        warnings.append(f'Active FTI parameter {fti_param} is not in infer_params.')

    return {
        'subfile': subfile,
        'config': config,
        'prior_params': prior_params,
        'run_params': run_params,
        'infer_params': infer_params,
        'list_params': list_params,
        'fisher_params': fisher_params,
        'fti_param': fti_param,
        'out_dir': out_dir,
        'out_name': out_name,
        'postfile': postfile,
        'post_raw_file': post_raw_file,
        'post_path_kind': post_path_kind,
        'raw_path_kind': raw_path_kind,
        'post_candidates': post_candidates,
        'raw_candidates': raw_candidates,
        'warnings': warnings,
    }


def print_case_paths(case):
    print('=' * 90)
    print('FTI parameter:', case['fti_param'])
    print('JSON:', case['subfile'])
    print('out_dir:', case['out_dir'])
    print('out_name:', case['out_name'])
    print('infer_params:', case['infer_params'])

    print('\nProcessed .h5:')
    if case['postfile'] is None:
        print('  not found')
        print('  tried:')
        for label, p in case['post_candidates']:
            print(f'    [{label}] {p}')
    else:
        print(f"  [{case['post_path_kind']}] {case['postfile']}")

    print('\nRaw _raw.h5:')
    if case['post_raw_file'] is None:
        print('  not found')
        print('  tried:')
        for label, p in case['raw_candidates']:
            print(f'    [{label}] {p}')
    else:
        print(f"  [{case['raw_path_kind']}] {case['post_raw_file']}")

    if case['warnings']:
        print('\nWarnings:')
        for w in case['warnings']:
            print('  -', w)
    print('')


def load_raw_from_file(rawfile, infer_params):
    """
    Load raw parameters and raw lnL from the _raw.h5 file.

    example_mbhb.ipynb uses:
        post_raw[p] = hf[p][:]
        post_raw['lnL'] = hf['lnlike'][:]
    """
    post_raw = {}

    with h5py.File(rawfile, 'r') as hf:
        for p in infer_params:
            if p in hf:
                post_raw[p] = hf[p][:]
            else:
                print(f"[warning] parameter {p} not found in raw file {rawfile}")

        if 'lnlike' in hf:
            post_raw['lnL'] = hf['lnlike'][:]
        elif 'lnL' in hf:
            post_raw['lnL'] = hf['lnL'][:]
        else:
            print('Available keys in raw file:')
            for k in hf.keys():
                print(' ', k)
            raise KeyError("Could not find 'lnlike' or 'lnL' in raw file.")

    return post_raw


def load_case(subfile):
    case = get_case_info(subfile)

    if case['postfile'] is not None:
        try:
            case['params_post'] = plotutils.load_params_posterior_lisa_smbh(
                str(case['subfile']),
                str(case['postfile'])
            )
        except Exception as exc:
            print(f"[warning] could not load processed posterior for {case['subfile'].name}: {exc}")
            case['params_post'] = None
            case['warnings'].append(f'Processed posterior exists but failed to load: {exc}')
    else:
        case['params_post'] = None

    if case['post_raw_file'] is not None:
        try:
            case['post_raw'] = load_raw_from_file(
                case['post_raw_file'],
                case['infer_params']
            )
        except Exception as exc:
            print(f"[warning] could not load raw posterior for {case['subfile'].name}: {exc}")
            case['post_raw'] = None
            case['warnings'].append(f'Raw posterior exists but failed to load: {exc}')
    else:
        case['post_raw'] = None

    return case


# ---------- Safe raw burn-in utilities ----------

def raw_nsteps(case):
    post_raw = case.get('post_raw')
    if post_raw is None or 'lnL' not in post_raw:
        return None
    return int(np.asarray(post_raw['lnL']).shape[-1])


def decide_raw_burnin(case):
    """
    Apply burn-in to raw files only when the raw length clearly equals n_iter.
    If raw length equals n_iter - burn_in, do not apply a second burn-in.
    """
    post_raw = case.get('post_raw')
    if post_raw is None or 'lnL' not in post_raw:
        return 0, 'no raw lnL available'

    nsteps = int(np.asarray(post_raw['lnL']).shape[-1])
    run_params = case.get('run_params', {})

    try:
        n_iter = int(run_params.get('n_iter')) if run_params.get('n_iter') is not None else None
    except Exception:
        n_iter = None

    try:
        burn_in = int(run_params.get('burn_in', 0) or 0)
    except Exception:
        burn_in = 0

    if burn_in <= 0:
        return 0, 'config burn_in is 0 or missing'
    if burn_in >= nsteps:
        return 0, f'not applying burn-in because burn_in={burn_in} >= raw nsteps={nsteps}'
    if n_iter is not None and nsteps == n_iter:
        return burn_in, f'applying burn-in because raw nsteps={nsteps} equals config n_iter={n_iter}'
    if n_iter is not None and nsteps == max(n_iter - burn_in, 0):
        return 0, f'not applying extra burn-in because raw nsteps={nsteps} equals n_iter - burn_in={n_iter - burn_in}'
    if n_iter is not None:
        return 0, f'ambiguous raw length: nsteps={nsteps}, n_iter={n_iter}, burn_in={burn_in}; no extra burn-in applied'
    return 0, 'config n_iter missing; no extra burn-in applied'


def slice_last_axis(arr, start=0, thin=1):
    arr = np.asarray(arr)
    index = [slice(None)] * arr.ndim
    index[-1] = slice(start, None, thin)
    return arr[tuple(index)]


def align_raw_array_to_lnL(arr, lnL):
    arr = np.asarray(arr)
    lnL = np.asarray(lnL)
    if arr.shape == lnL.shape:
        return arr
    if arr.size == lnL.size:
        return arr.reshape(lnL.shape)
    return None


def flatten_raw_samples(case, params=None, thin=1, remove_bad_lnl=True, max_samples=None, rng_seed=RANDOM_SEED):
    post_raw = case['post_raw']
    if post_raw is None:
        raise ValueError('No raw posterior loaded for this case.')

    if params is None:
        params = case['infer_params']

    burnin, burnin_reason = decide_raw_burnin(case)

    lnL_full = np.asarray(post_raw['lnL'])
    lnL = slice_last_axis(lnL_full, start=burnin, thin=thin)

    if remove_bad_lnl:
        mask = lnL > BAD_LNL_CUT
    else:
        mask = np.ones_like(lnL, dtype=bool)

    columns = []
    labels = []

    for p in params:
        if p not in post_raw:
            print(f"[warning] skipping {p}; not found in raw file")
            continue

        arr_full = align_raw_array_to_lnL(post_raw[p], lnL_full)
        if arr_full is None:
            print(f"[warning] skipping {p}; shape {np.asarray(post_raw[p]).shape} cannot be aligned with lnL shape {lnL_full.shape}")
            continue

        arr = slice_last_axis(arr_full, start=burnin, thin=thin)
        columns.append(arr[mask])
        labels.append(p)

    if len(columns) == 0:
        raise ValueError('No raw parameters could be flattened. Check raw file keys and shapes.')

    samples = np.vstack(columns).T
    lnL_flat = lnL[mask]

    if max_samples is not None and len(samples) > max_samples:
        rng = np.random.default_rng(rng_seed)
        idx = rng.choice(len(samples), size=max_samples, replace=False)
        samples = samples[idx]
        lnL_flat = lnL_flat[idx]

    meta = {
        'burnin_used': burnin,
        'burnin_reason': burnin_reason,
        'thin': thin,
        'n_samples': int(len(samples)),
        'params': labels,
    }

    return samples, labels, lnL_flat, meta

## Load all FTI cases

In [4]:
cases = []

for subfile in config_files:
    case = load_case(subfile)
    print_case_paths(case)
    cases.append(case)

cases_with_post = [c for c in cases if c['params_post'] is not None]
cases_with_raw = [c for c in cases if c['post_raw'] is not None]

print('Total JSON cases:', len(cases))
print('Cases with processed .h5:', len(cases_with_post))
print('Cases with raw _raw.h5:', len(cases_with_raw))


FTI parameter: fti_dchi0v
JSON: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/M1e6_SEOBv5_id_2_fti_dchi0v_IMR.json
out_dir: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_2
out_name: M1e6_SEOBv5_id_2_fti_dchi0v_IMR
infer_params: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi0v']

Processed .h5:
  [configured_string_concat] /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_2M1e6_SEOBv5_id_2_fti_dchi0v_IMR.h5

Raw _raw.h5:
  [configured_string_concat] /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_2M1e6_SEOBv5_id_2_fti_dchi0v_IMR_raw.h5

FTI parameter: fti_dchi1v
JSON: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/M1e6_SEOBv5_id_2_fti_dchi1v_IMR.json
out_dir: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_2
out_name: M1e6_SEOBv5_id_2_fti_dchi1v_IMR
infer_params: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi

## Diagnostics table

This table records which paths were used, whether the raw chain received extra burn-in, and whether the JSON looks internally consistent.

In [5]:
def case_summary_row(case):
    burnin_used, burnin_reason = decide_raw_burnin(case)
    rp = case.get('run_params', {})
    wp = case.get('config', {}).get('waveform_params', {})
    fti_block = wp.get('tgr_params', {}).get('FTI', {})

    return {
        'fti_case': case['fti_param'],
        'json_file': str(case['subfile']),
        'out_dir': str(case['out_dir']),
        'out_name': case['out_name'],
        'post_found': case['postfile'] is not None,
        'post_path_kind': case['post_path_kind'],
        'postfile': str(case['postfile']) if case['postfile'] is not None else None,
        'raw_found': case['post_raw_file'] is not None,
        'raw_path_kind': case['raw_path_kind'],
        'rawfile': str(case['post_raw_file']) if case['post_raw_file'] is not None else None,
        'sampler': rp.get('sampler'),
        'n_walkers': rp.get('n_walkers'),
        'n_temps': rp.get('n_temps'),
        'n_iter_config': rp.get('n_iter'),
        'burn_in_config': rp.get('burn_in'),
        'thin_samples_config': rp.get('thin_samples'),
        'raw_nsteps': raw_nsteps(case),
        'raw_burnin_used_by_notebook': burnin_used,
        'raw_burnin_reason': burnin_reason,
        'minf': wp.get('minf'),
        'maxf': wp.get('maxf'),
        'approximant': wp.get('approximant'),
        'TDI': wp.get('TDI'),
        'Mchirp_inj_FTI_block': fti_block.get('Mchirp_inj'),
        'dxi0v_inj_FTI_block': fti_block.get('dxi0v_inj'),
        'infer_params': ', '.join(case['infer_params']),
        'warnings': ' | '.join(case['warnings']),
    }

summary_table = pd.DataFrame([case_summary_row(c) for c in cases])
summary_csv = table_output_dir / 'fti_case_diagnostics_config_consistent.csv'
summary_table.to_csv(summary_csv, index=False)
print('Saved diagnostics table:', summary_csv)
summary_table

Saved diagnostics table: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/tables/fti_case_diagnostics_config_consistent.csv


,fti_case,json_file,out_dir,out_name,post_found,post_path_kind,postfile,raw_found,raw_path_kind,rawfile,...,raw_burnin_used_by_notebook,raw_burnin_reason,minf,maxf,approximant,TDI,Mchirp_inj_FTI_block,dxi0v_inj_FTI_block,infer_params,warnings
0,fti_dchi0v,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,M1e6_SEOBv5_id_2_fti_dchi0v_IMR,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,...,5000,applying burn-in because raw nsteps=10000 equa...,0.00001,0.2,SEOBNRv5HMROM,TDIAET,333021.282961,0.0,"Mchirp, q, chiPN, chim, Deltat, dist, inc, phi...",
1,fti_dchi1v,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,M1e6_SEOBv5_id_2_fti_dchi1v_IMR,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,...,5000,applying burn-in because raw nsteps=10000 equa...,0.00001,0.2,SEOBNRv5HMROM,TDIAET,333021.282961,0.0,"Mchirp, q, chiPN, chim, Deltat, dist, inc, phi...",
2,fti_dchi2v,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,M1e6_SEOBv5_id_2_fti_dchi2v_IMR,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,...,5000,applying burn-in because raw nsteps=10000 equa...,0.00001,0.2,SEOBNRv5HMROM,TDIAET,333021.282961,0.0,"Mchirp, q, chiPN, chim, Deltat, dist, inc, phi...",
3,fti_dchi3v,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,M1e6_SEOBv5_id_2_fti_dchi3v_IMR,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,...,5000,applying burn-in because raw nsteps=10000 equa...,0.00001,0.2,SEOBNRv5HMROM,TDIAET,333021.282961,0.0,"Mchirp, q, chiPN, chim, Deltat, dist, inc, phi...",
4,fti_dchi4v,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,M1e6_SEOBv5_id_2_fti_dchi4v_IMR,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,...,5000,applying burn-in because raw nsteps=10000 equa...,0.00001,0.2,SEOBNRv5HMROM,TDIAET,333021.282961,0.0,"Mchirp, q, chiPN, chim, Deltat, dist, inc, phi...",
5,fti_dchi5lv,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,M1e6_SEOBv5_id_2_fti_dchi5lv_IMR,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,...,5000,applying burn-in because raw nsteps=10000 equa...,0.00001,0.2,SEOBNRv5HMROM,TDIAET,333021.282961,0.0,"Mchirp, q, chiPN, chim, Deltat, dist, inc, phi...",
6,fti_dchi6lv,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,M1e6_SEOBv5_id_2_fti_dchi6lv_IMR,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,...,5000,applying burn-in because raw nsteps=10000 equa...,0.00001,0.2,SEOBNRv5HMROM,TDIAET,333021.282961,0.0,"Mchirp, q, chiPN, chim, Deltat, dist, inc, phi...",
7,fti_dchi6v,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,M1e6_SEOBv5_id_2_fti_dchi6v_IMR,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,True,configured_string_concat,/pbs/home/a/amahdi/PE_for_my_work/run_data/M1e...,...,5000,applying burn-in because raw nsteps=10000 equa...,0.00001,0.2,SEOBNRv5HMROM,TDIAET,333021.282961,0.0,"Mchirp, q, chiPN, chim, Deltat, dist, inc, phi...",
8,fti_dch

## Likelihood histogram from processed posterior

Same style as `example_mbhb.ipynb`:

`ax.hist(params_post['post']['lnL'], histtype='step', density=True)`


In [6]:
if len(cases_with_post) == 0:
    print('No processed .h5 files found. Skipping posterior likelihood histogram.')
else:
    fig, ax = plt.subplots(1, 1, figsize=[6, 4])

    for case in cases_with_post:
        params_post = case['params_post']
        fti_param = case['fti_param']

        ax.hist(
            params_post['post']['lnL'],
            histtype='step',
            density=True,
            label=fti_param
        )

    ax.set_xlabel(r'$\ln \mathcal{L}$')
    ax.set_ylabel('density')
    ax.legend(fontsize=8)

    fig.tight_layout()
    outfile = likelihood_output_dir / 'posterior_likelihood_histogram_all_fti.png'
    fig.savefig(outfile, dpi=200, bbox_inches='tight')
    print('Saved:', outfile)
    display(fig)
    plt.close(fig)


Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/likelihood/posterior_likelihood_histogram_all_fti.png


<Figure size 432x288 with 1 Axes>

## Raw likelihood trace

Same structure as `example_mbhb.ipynb`.


In [7]:
def plot_postraw_lnL_trace(case, rangey=None):
    post_raw = case['post_raw']
    lnL = np.asarray(post_raw['lnL'])
    nsteps = lnL.shape[-1]
    burnin_used, burnin_reason = decide_raw_burnin(case)

    # Flatten all leading axes, such as walkers or temperature/walker axes.
    lnL_2d = lnL.reshape(-1, nsteps)
    steps = np.arange(nsteps)

    fig, ax = plt.subplots(1, 1, figsize=[16, 4])

    series = []
    for i in range(lnL_2d.shape[0]):
        mask = lnL_2d[i] > BAD_LNL_CUT
        if np.any(mask):
            series.append([steps[mask], -lnL_2d[i][mask]])

    if len(series) > 0:
        plotutils.llogplot(ax, *series, rangey=rangey)
    else:
        ax.plot([], [])
        print(f"[warning] no valid lnL values for {case['fti_param']}")

    if burnin_used > 0:
        ax.axvline(burnin_used, linestyle='--', linewidth=1)

    ax.set_xlabel(r'$\mathrm{Steps}$')
    ax.set_ylabel(r'$-\ln \mathcal{L}$')
    ax.set_title(f"{case['fti_param']}\nraw burn-in used: {burnin_used}; {burnin_reason}")

    return fig

In [8]:
if len(cases_with_raw) == 0:
    print('No raw _raw.h5 files found. Nothing to plot.')
else:
    for case in cases_with_raw:
        print('Plotting raw trace:', case['fti_param'])
        fig = plot_postraw_lnL_trace(case)

        outfile = trace_output_dir / f"{case['fti_param']}_raw_likelihood_trace.png"
        fig.savefig(outfile, dpi=200, bbox_inches='tight')
        print('Saved:', outfile)

        display(fig)
        plt.close(fig)

Plotting raw trace: fti_dchi0v
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/trace/fti_dchi0v_raw_likelihood_trace.png


<Figure size 1600x400 with 1 Axes>

Plotting raw trace: fti_dchi1v


/pbs/home/a/amahdi/lisabeta/src/lisabeta/utils/plotutils.py:120: UserWarning: Ignoring specified arguments in this call because figure with num: 0 already exists
  f = plt.figure(0, figsize=size)


Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/trace/fti_dchi1v_raw_likelihood_trace.png


...

<Figure size 1600x400 with 1 Axes>

Plotting raw trace: fti_dkappa_S
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/trace/fti_dkappa_S_raw_likelihood_trace.png


<Figure size 1600x400 with 1 Axes>

Plotting raw trace: fti_dxi0v
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/trace/fti_dxi0v_raw_likelihood_trace.png


<Figure size 1600x400 with 1 Axes>

# Corner plots

First choice: if processed `.h5` exists, use `plotutils.corner_plot(...)`.

Fallback: if only `_raw.h5` exists, flatten raw parameter chains using the safe burn-in logic above and use `corner.corner(...)`.

The full corner uses `case["infer_params"]`, which is exactly the list written in the JSON config.

In [9]:
def unique_param_names(params):
    """Return parameter names in their original order without duplicates."""
    return list(dict.fromkeys(params))


def force_active_fti_param(case, params=None):
    """
    Always append the active FTI parameter to the requested corner parameters.

    Each JSON run samples one active FTI parameter. Therefore each corner plot
    contains the ordinary sampled parameters plus that run's active FTI parameter.
    """
    if params is None:
        params = list(case['infer_params'])
    else:
        params = list(params)

    fti_param = case.get('fti_param')
    if fti_param and fti_param not in params:
        params.append(fti_param)

    return unique_param_names(params)


def post_has_param(post, p):
    try:
        return p in post
    except Exception:
        return False


def raw_has_param(post_raw, p):
    try:
        return post_raw is not None and p in post_raw
    except Exception:
        return False


def available_processed_params(case, params=None):
    params = force_active_fti_param(case, params)
    if case.get('params_post') is None:
        return []

    post = case['params_post']['post']
    return [p for p in params if post_has_param(post, p)]


def available_raw_params(case, params=None):
    params = force_active_fti_param(case, params)
    post_raw = case.get('post_raw')
    return [p for p in params if raw_has_param(post_raw, p)]


def choose_corner_source(case, params=None):
    """
    Select a posterior source without silently dropping the active FTI parameter.

    Preference:
      1. processed posterior, only when it contains every requested parameter,
         including the active FTI parameter;
      2. raw chain, when it contains the active FTI parameter;
      3. processed posterior containing the active FTI parameter, even if another
         requested parameter is unavailable.

    A source that does not contain the active FTI parameter is never selected.
    """
    requested = force_active_fti_param(case, params)
    fti_param = case.get('fti_param')

    processed = available_processed_params(case, requested)
    raw = available_raw_params(case, requested)

    if (
        case.get('params_post') is not None
        and fti_param in processed
        and len(processed) == len(requested)
    ):
        return 'processed', processed

    if case.get('post_raw') is not None and fti_param in raw:
        missing = [p for p in requested if p not in raw]
        if missing:
            print(f"[warning] raw corner for {fti_param} is missing: {missing}")
        if case.get('params_post') is not None and fti_param not in processed:
            print(
                f"[warning] {fti_param} is absent from the processed posterior; "
                "using the raw chain so the FTI parameter is retained."
            )
        return 'raw', raw

    if case.get('params_post') is not None and fti_param in processed:
        missing = [p for p in requested if p not in processed]
        if missing:
            print(f"[warning] processed corner for {fti_param} is missing: {missing}")
        return 'processed', processed

    print(
        f"[error] Cannot make an FTI-inclusive corner for {fti_param}. "
        "The active FTI parameter is absent from both the processed posterior "
        "and the loaded raw chain."
    )
    print('  requested:', requested)
    print('  processed available:', processed)
    print('  raw available:', raw)
    return None, []


def inj_has_params(params_post, params):
    inj = params_post.get('injparams_Lframe', {})
    try:
        return all(p in inj for p in params)
    except Exception:
        return False


def raw_truth_values(case, labels):
    """
    Use processed injection values when available, otherwise source_params.
    Return None unless every plotted parameter has a known injected value.
    """
    truth_sources = []

    if case.get('params_post') is not None:
        truth_sources.append(case['params_post'].get('injparams_Lframe', {}))

    truth_sources.append(case.get('config', {}).get('source_params', {}))

    truths = []
    for p in labels:
        found = False
        for source in truth_sources:
            try:
                if p in source:
                    truths.append(source[p])
                    found = True
                    break
            except Exception:
                pass
        if not found:
            return None

    return truths


def plot_corner_from_processed(case, params=None, bins=25, use_fisher=False, plot_datapoints=False):
    """
    Plot a processed posterior while explicitly requiring the active FTI parameter.
    """
    params_post = case['params_post']
    requested = force_active_fti_param(case, params)
    available = available_processed_params(case, requested)
    fti_param = case['fti_param']

    if fti_param not in available:
        print(
            f"[warning] refusing processed corner because {fti_param} is absent. "
            "Use plot_corner_auto() to fall back to the raw chain."
        )
        return None

    if len(available) < 2:
        print(f"Not enough processed posterior parameters for {fti_param}: {available}")
        return None

    missing = [p for p in requested if p not in available]
    if missing:
        print(f"[warning] processed posterior is missing: {missing}")

    show_truths = inj_has_params(params_post, available)

    kwargs = dict(
        output=False,
        histograms=True,
        params=available,
        Lframe=True,
        color=plotutils.plotpalette[0],
        show_truths=show_truths,
        truth_color='k',
        bins=bins,
        show_histograms=True,
        plot_datapoints=plot_datapoints,
        fisher=False,
    )

    if use_fisher:
        fishercov = params_post.get('fishercov', None)
        if fishercov is not None:
            kwargs['fisher'] = True
            kwargs['fishercov'] = fishercov
            kwargs['cov_color'] = 'k'
        else:
            print(f"[warning] requested Fisher overlay but fishercov is missing for {fti_param}")

    print('Processed corner parameters:', available)

    fig = plotutils.corner_plot(
        params_post['injparams_Lframe'],
        params_post['post'],
        **kwargs
    )

    fig.suptitle(f"{fti_param} processed posterior", y=1.02)
    return fig


def plot_corner_from_raw(case, params=None, bins=25, max_samples=MAX_RAW_SAMPLES_FOR_CORNER):
    """
    Plot the raw chain, explicitly retaining the active FTI parameter.
    """
    if not HAS_CORNER:
        raise ImportError(
            "The processed posterior does not contain the active FTI parameter "
            "and the Python package 'corner' is unavailable. Install it with: "
            "pip install corner"
        )

    requested = force_active_fti_param(case, params)

    samples, labels, lnL_flat, meta = flatten_raw_samples(
        case,
        params=requested,
        thin=1,
        remove_bad_lnl=True,
        max_samples=max_samples,
    )

    fti_param = case['fti_param']
    if fti_param not in labels:
        print(
            f"[error] refusing raw corner because the active FTI parameter "
            f"{fti_param} was not loaded. Available labels: {labels}"
        )
        return None

    if len(labels) < 2:
        print(f"Not enough raw parameters for {fti_param}: {labels}")
        return None

    truths = raw_truth_values(case, labels)

    print('Raw corner parameters:', labels)

    figure = corner.corner(
    normalized_for_plot,

    labels=labels,
    truths=truths,
    truth_color='k',

    color=case_colour,

    # More bins make the histogram cells smaller
    bins=60,

    # Smooth the 2D and 1D posterior histograms
    smooth=1.0,
    smooth1d=1.0,

    show_titles=True,
    title_fmt='.2f',
    quantiles=[0.16, 0.50, 0.84],

    # Do not show individual samples as discrete points
    plot_datapoints=False,

    plot_density=True,
    plot_contours=True,
    fill_contours=True,

    max_n_ticks=3,

    label_kwargs={
        'fontsize': 8,
    },

    title_kwargs={
        'fontsize': 8,
    },

    hist_kwargs={
        'color': case_colour,
        'linewidth': 1.4,
        'density': True,
    },

    contour_kwargs={
        'colors': [case_colour],
        'linewidths': 1.2,
    },

    contourf_kwargs={
        'alpha': 0.55,
    },

    levels=(
        1.0 - np.exp(-0.5),
        1.0 - np.exp(-2.0),
    ),
)

    fig.suptitle(
        f"{fti_param} raw posterior\n"
        f"raw burn-in used: {meta['burnin_used']}; {meta['burnin_reason']}",
        y=1.02,
    )

    return fig


def plot_corner_auto(case, params=None, bins=25, use_fisher=False, plot_datapoints=False):
    """
    Make a corner plot from the best source that contains the active FTI parameter.
    """
    requested = force_active_fti_param(case, params)
    source, selected = choose_corner_source(case, requested)

    print('\nFTI case:', case['fti_param'])
    print('Requested corner parameters:', requested)
    print('Selected corner source:', source)
    print('Selected corner parameters:', selected)

    if source == 'processed':
        return plot_corner_from_processed(
            case,
            params=selected,
            bins=bins,
            use_fisher=use_fisher,
            plot_datapoints=plot_datapoints,
        ), source

    if source == 'raw':
        return plot_corner_from_raw(
            case,
            params=selected,
            bins=bins,
        ), source

    return None, None


## Corner plot for each FTI case

This uses processed posterior if available. Otherwise it uses raw samples.


In [10]:
for case in cases:
    # force_active_fti_param() guarantees that the active FTI parameter is requested.
    params_full = force_active_fti_param(case, case['infer_params'])

    fig, corner_source = plot_corner_auto(
        case,
        params=params_full,
        bins=25,
        plot_datapoints=False,
    )

    outfile = corner_output_dir / (
        f"{case['fti_param']}_full_infer_params_with_fti_{corner_source}.png"
    )

    if fig is not None:
        fig.savefig(outfile, dpi=200, bbox_inches='tight')
        print('Saved:', outfile)
        display(fig)
        plt.close(fig)
    else:
        print('No FTI-inclusive corner plot produced for:', case['fti_param'])



FTI case: fti_dchi0v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi0v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi0v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi0v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_dchi0v_full_infer_params_with_fti_processed.png


<Figure size 2650x2650 with 144 Axes>


FTI case: fti_dchi1v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi1v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi1v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi1v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_dchi1v_full_infer_params_with_fti_processed.png


<Figure size 2650x2650 with 144 Axes>


FTI case: fti_dchi2v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi2v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi2v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi2v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_dchi2v_full_infer_params_with_fti_processed.png


...

<Figure size 2650x2650 with 144 Axes>


FTI case: fti_dkappa_S
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dkappa_S']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dkappa_S']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dkappa_S']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_dkappa_S_full_infer_params_with_fti_processed.png


<Figure size 2650x2650 with 144 Axes>


FTI case: fti_dxi0v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dxi0v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dxi0v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dxi0v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_dxi0v_full_infer_params_with_fti_processed.png


<Figure size 2650x2650 with 144 Axes>

# Maximum-likelihood parameter values

This prints the parameter values at the sample with largest `lnL`.

For processed posterior `.h5`, it uses `params_post['post']['lnL']` directly.

For raw `_raw.h5`, it uses the safe raw burn-in logic above. It does not blindly apply a second burn-in if the raw file already looks post-burn-in.

In [11]:
def max_likelihood_values_from_processed(case):
    params_post = case['params_post']
    post = params_post['post']
    params = case['infer_params']

    lnL = np.asarray(post['lnL'])
    imax = int(np.argmax(lnL))

    row = {
        'fti_case': case['fti_param'],
        'source': 'processed',
        'lnL_max': float(lnL[imax]),
        'sample_index': imax,
        'raw_burnin_used': None,
        'raw_burnin_reason': None,
    }

    for p in params:
        if p in post:
            row[p] = np.asarray(post[p])[imax]

    return row


def max_likelihood_values_from_raw(case):
    params = case['infer_params']
    samples, labels, lnL_flat, meta = flatten_raw_samples(
        case,
        params=params,
        thin=1,
        remove_bad_lnl=True,
        max_samples=None,
    )

    imax = int(np.argmax(lnL_flat))

    row = {
        'fti_case': case['fti_param'],
        'source': 'raw',
        'lnL_max': float(lnL_flat[imax]),
        'sample_index': imax,
        'raw_burnin_used': meta['burnin_used'],
        'raw_burnin_reason': meta['burnin_reason'],
    }

    for j, p in enumerate(labels):
        row[p] = samples[imax, j]

    return row


ml_rows = []

for case in cases:
    try:
        if case['params_post'] is not None:
            ml_rows.append(max_likelihood_values_from_processed(case))
        elif case['post_raw'] is not None:
            ml_rows.append(max_likelihood_values_from_raw(case))
        else:
            print('No data for maximum-likelihood values:', case['fti_param'])
    except Exception as exc:
        print(f"[warning] failed to compute ML values for {case['fti_param']}: {exc}")

ml_table = pd.DataFrame(ml_rows)

ml_table

,fti_case,source,lnL_max,sample_index,raw_burnin_used,raw_burnin_reason,Mchirp,q,chiPN,chim,...,fti_dchi2v,fti_dchi3v,fti_dchi4v,fti_dchi5lv,fti_dchi6lv,fti_dchi6v,fti_dchi7v,fti_dchiMinus2v,fti_dkappa_S,fti_dxi0v
0,fti_dchi0v,processed,-1.109292,27412,None,None,333059.577728,3.993893,0.445888,0.299506,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fti_dchi1v,processed,-1.049492,25254,None,None,333023.394874,3.994640,0.445936,0.299071,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,fti_dchi2v,processed,-1.005413,29064,None,None,333020.414160,3.993498,0.445888,0.299070,...,0.000009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,fti_dchi3v,processed,-1.316563,2375,None,None,333020.566110,3.995158,0.445814,0.300340,...,NaN,0.000115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,fti_dchi4v,processed,-1.237479,20066,None,None,333017.903528,3.990892,0.445690,0.299723,...,NaN,NaN,0.001222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,fti_dchi5lv,processed,-0.947152,36637,None,None,333021.270952,3.991258,0.445938,0.297627,...,NaN,NaN,NaN,-0.001289,NaN,NaN,NaN,NaN,NaN,NaN
6,fti_dchi6lv,processed,-0.785971,3574,None,None,333019.209798,3.993713,0.445895,0.299071,...,NaN,NaN,NaN,NaN,-0.001597,NaN,NaN,NaN,NaN,NaN
7,fti_dchi6v,processed,-1.315544,26937,None,None,333019.788016,3.996063,0.446038,0.298758,...,NaN,NaN,NaN,NaN,NaN,0.000588,NaN,NaN,NaN,NaN
8,fti_dchi7v,processed,-1.140326,18083,None,None,333019.439061,3.995005,0.445877,0.299831,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000419,NaN,NaN,NaN
9,fti_dchiMinus2v,processed,-1.292130,21116,None,None,333017.812201,3.994302,0.445778,0.300329,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.593373e-08,NaN,NaN


## Save maximum-likelihood table to CSV

In [12]:
output_csv = table_output_dir / 'fti_max_likelihood_values_config_consistent.csv'

if len(ml_table) > 0:
    ml_table.to_csv(output_csv, index=False)
    print('Saved:', output_csv)
else:
    print('No maximum-likelihood values to save.')

Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/tables/fti_max_likelihood_values_config_consistent.csv


## Inspect one raw file

Use this only if something fails and you want to see the dataset names.


In [13]:
def h5_dataset_summary(h5file):
    rows = []
    with h5py.File(h5file, 'r') as hf:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                rows.append({'name': name, 'shape': obj.shape, 'dtype': str(obj.dtype)})
        hf.visititems(visit)
    return pd.DataFrame(rows)

if len(cases_with_raw) > 0:
    rawfile = cases_with_raw[0]['post_raw_file']
    print('Inspecting:', rawfile)
    display(h5_dataset_summary(rawfile))
else:
    print('No raw files found.')

Inspecting: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_2M1e6_SEOBv5_id_2_fti_dchi0v_IMR_raw.h5


,name,shape,dtype
0,Deltat,"(64, 10000)",float64
1,Mchirp,"(64, 10000)",float64
2,beta,"(64, 10000)",float64
3,chiPN,"(64, 10000)",float64
4,chim,"(64, 10000)",float64
5,dist,"(64, 10000)",float64
6,fti_dchi0v,"(64, 10000)",float64
7,inc,"(64, 10000)",float64
8,lambda,"(64, 10000)",float64
9,lnlike,"(64, 10000)",float64


# Smaller config-consistent corner plots

These replace the old cells that used `M`, `m1`, `m2`, `chi1`, `chi2`, or `iota` as default parameters.

The default small corner now uses sampled coordinates only:

`Mchirp, q, chiPN, chim, dist, inc, active_FTI_param`

In [14]:
small_corner_output_dir = corner_output_dir / 'small_parameter_subset'
small_corner_output_dir.mkdir(parents=True, exist_ok=True)


def choose_small_param_set(case, preferred_params=None, max_params=7):
    infer_params = list(case['infer_params'])
    fti_param = case['fti_param']

    if preferred_params is None:
        preferred_params = SAMPLED_EXTENDED_PARAMS.copy()

    params_small = [p for p in preferred_params if p in infer_params]

    # This is unconditional with respect to the preferred list: the active FTI
    # parameter must be present in every small corner plot.
    if fti_param and fti_param not in params_small:
        params_small.append(fti_param)

    if max_params is not None and len(params_small) > max_params:
        non_fti = [p for p in params_small if p != fti_param]
        params_small = non_fti[:max_params - 1] + [fti_param]

    return force_active_fti_param(case, params_small)


small_preferred_params = ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'lambda', 'beta']

for case in cases:
    params_small = choose_small_param_set(
        case,
        preferred_params=small_preferred_params,
        max_params=7,
    )

    fig, corner_source = plot_corner_auto(
        case,
        params=params_small,
        bins=25,
        plot_datapoints=False,
    )

    if fig is not None:
        outfile = small_corner_output_dir / (
            f"{case['fti_param']}_small_sampled_params_with_fti_{corner_source}.png"
        )
        fig.savefig(outfile, dpi=200, bbox_inches='tight')
        print('Saved:', outfile)
        display(fig)
        plt.close(fig)



FTI case: fti_dchi0v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi0v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi0v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi0v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/small_parameter_subset/fti_dchi0v_small_sampled_params_with_fti_processed.png


<Figure size 1600x1600 with 49 Axes>


FTI case: fti_dchi1v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi1v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi1v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi1v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/small_parameter_subset/fti_dchi1v_small_sampled_params_with_fti_processed.png


<Figure size 1600x1600 with 49 Axes>


FTI case: fti_dchi2v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi2v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi2v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dchi2v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/small_parameter_subset/fti_dchi2v_small_sampled_params_with_fti_processed.png


...

<Figure size 1600x1600 with 49 Axes>


FTI case: fti_dkappa_S
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dkappa_S']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dkappa_S']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dkappa_S']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/small_parameter_subset/fti_dkappa_S_small_sampled_params_with_fti_processed.png


<Figure size 1600x1600 with 49 Axes>


FTI case: fti_dxi0v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dxi0v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dxi0v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'dist', 'inc', 'fti_dxi0v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/small_parameter_subset/fti_dxi0v_small_sampled_params_with_fti_processed.png


<Figure size 1600x1600 with 49 Axes>

# FTI-focused corner plots for every FTI parameter

For each JSON case, the focused plot uses:

`Mchirp, q, chiPN, chim, active_FTI_parameter`

Processed posterior data are preferred. If only a raw file exists, the notebook uses the raw-chain fallback with the safe burn-in logic.


In [15]:
fti_focused_output_dir = corner_output_dir / 'fti_focused'
fti_focused_output_dir.mkdir(parents=True, exist_ok=True)


def choose_fti_focused_params(case, params_core=None):
    if params_core is None:
        params_core = SAMPLED_INTRINSIC_PARAMS.copy()

    params_to_plot = [p for p in params_core if p in case['infer_params']]
    return force_active_fti_param(case, params_to_plot)


def plot_fti_focused_corner(case, params_core=None, bins=25):
    params_to_plot = choose_fti_focused_params(case, params_core=params_core)

    fig, corner_source = plot_corner_auto(
        case,
        params=params_to_plot,
        bins=bins,
        plot_datapoints=True,
    )

    if fig is not None:
        fig.suptitle(
            f"{case['fti_param']}, focused sampled parameters "
            f"with active FTI ({corner_source})",
            y=1.02,
        )

    return fig, corner_source


# Run one focused corner for every FTI JSON case.
for case in cases:
    fig, corner_source = plot_fti_focused_corner(
        case,
        params_core=['Mchirp', 'q', 'chiPN', 'chim'],
        bins=25,
    )

    if fig is not None:
        outfile = fti_focused_output_dir / (
            f"{case['fti_param']}_focused_with_fti_{corner_source}.png"
        )
        fig.savefig(outfile, dpi=200, bbox_inches='tight')
        print('Saved:', outfile)
        display(fig)
        plt.close(fig)



FTI case: fti_dchi0v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi0v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi0v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi0v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_focused/fti_dchi0v_focused_with_fti_processed.png


<Figure size 1180x1180 with 25 Axes>


FTI case: fti_dchi1v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi1v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi1v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi1v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_focused/fti_dchi1v_focused_with_fti_processed.png


<Figure size 1180x1180 with 25 Axes>


FTI case: fti_dchi2v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi2v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi2v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dchi2v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_focused/fti_dchi2v_focused_with_fti_processed.png


...

<Figure size 1180x1180 with 25 Axes>


FTI case: fti_dkappa_S
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dkappa_S']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dkappa_S']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dkappa_S']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_focused/fti_dkappa_S_focused_with_fti_processed.png


<Figure size 1180x1180 with 25 Axes>


FTI case: fti_dxi0v
Requested corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dxi0v']
Selected corner source: processed
Selected corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dxi0v']
Processed corner parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'fti_dxi0v']
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/corner/fti_focused/fti_dxi0v_focused_with_fti_processed.png


<Figure size 1180x1180 with 25 Axes>

# JSON consistency checks

This table checks whether the active FTI parameter is consistently placed in `source_params`, `list_params`, `infer_params`, and Fisher initialization parameters.

In [16]:
def config_consistency_row(case):
    config = case['config']
    fti = case['fti_param']
    source_params = config.get('source_params', {})
    fti_params_block = config.get('waveform_params', {}).get('tgr_params', {}).get('FTI', {}).get('fti_params', [])

    return {
        'fti_case': fti,
        'fti_in_source_params': fti in source_params,
        'fti_source_value': source_params.get(fti),
        'fti_in_infer_params': fti in case['infer_params'],
        'fti_is_last_infer_param': len(case['infer_params']) > 0 and case['infer_params'][-1] == fti,
        'fti_in_list_params': fti in case['list_params'],
        'fti_is_last_list_param': len(case['list_params']) > 0 and case['list_params'][-1] == fti,
        'fti_in_fisher_params': fti in case['fisher_params'],
        'fti_is_last_fisher_param': len(case['fisher_params']) > 0 and case['fisher_params'][-1] == fti,
        'fti_in_FTI_block_list': fti in fti_params_block,
        'only_one_fti_in_source_params': sum(1 for p in fti_params_block if p in source_params) == 1,
        'number_of_infer_params': len(case['infer_params']),
        'number_of_fisher_params': len(case['fisher_params']),
    }

consistency_table = pd.DataFrame([config_consistency_row(c) for c in cases])
consistency_csv = table_output_dir / 'fti_json_consistency_checks.csv'
consistency_table.to_csv(consistency_csv, index=False)
print('Saved:', consistency_csv)
consistency_table

Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_2/fti_plots_all_parameters_concat_path/tables/fti_json_consistency_checks.csv


,fti_case,fti_in_source_params,fti_source_value,fti_in_infer_params,fti_is_last_infer_param,fti_in_list_params,fti_is_last_list_param,fti_in_fisher_params,fti_is_last_fisher_param,fti_in_FTI_block_list,only_one_fti_in_source_params,number_of_infer_params,number_of_fisher_params
0,fti_dchi0v,True,0,True,True,True,True,True,True,True,True,12,12
1,fti_dchi1v,True,0,True,True,True,True,True,True,True,True,12,12
2,fti_dchi2v,True,0,True,True,True,True,True,True,True,True,12,12
3,fti_dchi3v,True,0,True,True,True,True,True,True,True,True,12,12
4,fti_dchi4v,True,0,True,True,True,True,True,True,True,True,12,12
5,fti_dchi5lv,True,0,True,True,True,True,True,True,True,True,12,12
6,fti_dchi6lv,True,0,True,True,True,True,True,True,True,True,12,12
7,fti_dchi6v,True,0,True,True,True,True,True,True,True,True,12,12
8,fti_dchi7v,True,0,True,True,True,True,True,True,True,True,12,12
9,fti_dchiMinus2v,True,0,True,True,True,True,True,True,True,True,12,12


# Notes

- `CASE_ID` is currently `M1e6_SEOBv5_id_3`.
- Every JSON file in that config directory is processed, so all available FTI parameters are included automatically.
- The confirmed output convention is string concatenation, such as `run_data/M1e6_SEOBv5_id_3M1e6_SEOBv5_id_3_fti_dchi0v_IMR_raw.h5`; this is searched before ordinary joined paths.
- `Mchirp`, `q`, `chiPN`, and `chim` are the sampled intrinsic coordinates in the current config setup.
- `M`, `m1`, `m2`, `chi1`, and `chi2` may be derived quantities in a processed posterior, but they are not assumed to be sampled coordinates.
- Processed `.h5` files are preferred for final plots and maximum-likelihood summaries.
- Raw `_raw.h5` files are supported as a diagnostic fallback, with burn-in applied only when the raw length clearly equals the configured `n_iter`.
- Full, small, and FTI-focused corner plots now all run for every detected FTI case.
